In [2]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json


In [16]:
class CaribooWildfirePipeline:
    """
    Complete data pipeline for Cariboo Fire Centre wildfire prediction
    Supports 4 fire zones with zone-specific weather and analysis
    """

    def __init__(self):
        self.weather_api = "https://archive-api.open-meteo.com/v1/archive"

        self.fire_zones = {
            'Central Cariboo': {
                'description': 'Administrative hub, mixed terrain',
                'primary_weather': (52.1411, -122.1430),      # Williams Lake
                'secondary_weather': (51.5167, -121.2000),    # Lac La Hache
                'elevation': 650,
                'bounds': {
                    'north': 52.5, 'south': 51.5, 
                    'east': -121.5, 'west': -122.5
                },
                'fire_season': 'May-October',
                'avg_fires_year': 100,
                'avg_fire_size_ha': 18,
                'characteristics': 'Mixed terrain, longest season, most fires'
            },
            '100 Mile House': {
                'description': 'Grassland-pine mix, lower elevation',
                'primary_weather': (51.6403, -121.2957),      # 100 Mile House
                'secondary_weather': (50.0252, -121.3021),    # Cache Creek
                'elevation': 960,
                'bounds': {
                    'north': 52.0, 'south': 51.0,
                    'east': -121.0, 'west': -122.0
                },
                'fire_season': 'May-September',
                'avg_fires_year': 70,
                'avg_fire_size_ha': 12,
                'characteristics': 'Grassland, smaller fires, easier suppression'
            },
            'Chilcotin': {
                'description': 'Remote, mountainous, heavy forest',
                'primary_weather': (52.4167, -121.5000),      # Chilanko Forks
                'secondary_weather': (52.3884, -126.3718),    # Bella Coola
                'elevation': 1200,
                'bounds': {
                    'north': 53.0, 'south': 51.8,
                    'east': -120.5, 'west': -127.0
                },
                'fire_season': 'June-August',
                'avg_fires_year': 50,
                'avg_fire_size_ha': 28,
                'characteristics': 'Remote, difficult access, larger fires'
            },
            'Quesnel': {
                'description': 'Boreal forest, northern, dense timber',
                'primary_weather': (52.9707, -122.4926),      # Quesnel
                'secondary_weather': (53.1667, -123.0000),    # Nazko
                'elevation': 670,
                'bounds': {
                    'north': 53.5, 'south': 52.5,
                    'east': -122.0, 'west': -123.5
                },
                'fire_season': 'June-September',
                'avg_fires_year': 60,
                'avg_fire_size_ha': 15,
                'characteristics': 'Boreal forest, northern patterns'
            }
        }
    def classify_fire_zone(self, latitude, longitude):
        """
        Classify a fire into one of the four zones based on coordinates

        Parameters:
        - latitude: Fire latitude (decimal degrees)
        - longitude: Fire longitude (decimal degrees)

        Returns: Zone name string
        """

        for zone_name, zone_info in self.fire_zones.items():
            bounds = zone_info['bounds']

            if (bounds['south'] <= latitude <= bounds['north'] and
                bounds['west'] <= longitude <= bounds['east']):
                return zone_name

        # If doesn't match any zone exactly, assign to nearest
        return self._assign_nearest_zone(latitude, longitude)
    def _assign_nearest_zone(self, lat, lon):
        """Assign to nearest zone center if outside all boundaries"""
        min_dist = float('inf')
        nearest_zone = 'Central Cariboo'

        for zone_name, zone_info in self.fire_zones.items():
            center_lat, center_lon = zone_info['primary_weather']
            dist = np.sqrt((lat - center_lat)**2 + (lon - center_lon)**2)

            if dist < min_dist:
                min_dist = dist
                nearest_zone = zone_name

        return nearest_zone

    def add_zone_classification(self, fires_df):
        """
        Add Fire_Zone column to fires dataframe

        Requires: Latitude, Longitude columns
        """
        if 'Fire_Zone' not in fires_df.columns:
            fires_df['Fire_Zone'] = fires_df.apply(
                lambda row: self.classify_fire_zone(
                    row['Latitude'], 
                    row['Longitude']
                ),
                axis=1
            )

        return fires_df
        # ==================== WEATHER DATA FETCHING ====================

    def fetch_weather(self, zone_name, start_date='2012-01-01', 
                     end_date='2017-12-31', use_secondary=False, verbose=True):
        """
        Fetch historical weather data for specific zone

        Parameters:
        - zone_name: One of the four fire zones
        - start_date: Start date (YYYY-MM-DD)
        - end_date: End date (YYYY-MM-DD)
        - use_secondary: Use secondary weather station if True
        - verbose: Print progress if True

        Returns: pandas DataFrame with daily weather
        """

        if zone_name not in self.fire_zones:
            raise ValueError(f"Unknown zone: {zone_name}. Must be one of: {list(self.fire_zones.keys())}")

        zone_info = self.fire_zones[zone_name]
        lat, lon = zone_info['secondary_weather'] if use_secondary else zone_info['primary_weather']

        if verbose:
            print(f"\n{'─'*70}")
            print(f"Fetching Weather: {zone_name} Fire Zone")
            print(f"{'─'*70}")
            print(f"  Location: {zone_info['description']}")
            print(f"  Coordinates: ({lat:.4f}, {lon:.4f})")
            print(f"  Elevation: {zone_info['elevation']}m")
            print(f"  Fire Season: {zone_info['fire_season']}")
            print(f"  Date Range: {start_date} to {end_date}")

        params = {
            'latitude': lat,
            'longitude': lon,
            'start_date': start_date,
            'end_date': end_date,
            'daily': [
                'temperature_2m_max',
                'temperature_2m_mean',
                'temperature_2m_min',
                'relative_humidity_2m_max',
                'relative_humidity_2m_mean',
                'relative_humidity_2m_min',
                'precipitation_sum',
                'wind_speed_10m_max',
                'pressure_msl_max',
                'dew_point_2m_mean',
            ],
            'temperature_unit': 'celsius',
            'wind_speed_unit': 'kmh',
            'precipitation_unit': 'mm',
            'timezone': 'America/Vancouver'
        }

        try:
            response = requests.get(self.weather_api, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()

            daily = data['daily']
            df = pd.DataFrame({
                'Date': pd.to_datetime(daily['time']),
                'Temp_Max_C': daily['temperature_2m_max'],
                'Temp_Mean_C': daily['temperature_2m_mean'],
                'Temp_Min_C': daily['temperature_2m_min'],
                'Humidity_Max_Pct': daily['relative_humidity_2m_max'],
                'Humidity_Mean_Pct': daily['relative_humidity_2m_mean'],
                'Humidity_Min_Pct': daily['relative_humidity_2m_min'],
                'Precip_mm': daily['precipitation_sum'],
                'Wind_Speed_Max_kmh': daily['wind_speed_10m_max'],
                'Pressure_Max_hPa': daily['pressure_msl_max'],
                'Dew_Point_Mean_C': daily['dew_point_2m_mean'],
                'Fire_Zone': zone_name,
                'Elevation_m': zone_info['elevation']
            })

            if verbose:
                print(f"  ✓ Downloaded {len(df)} daily records")
                print(f"  Weather Stats:")
                print(f"    Avg Temp: {df['Temp_Mean_C'].mean():.1f}°C (range: {df['Temp_Min_C'].min():.1f} to {df['Temp_Max_C'].max():.1f}°C)")
                print(f"    Avg Humidity: {df['Humidity_Mean_Pct'].mean():.1f}%")
                print(f"    Total Precip: {df['Precip_mm'].sum():.0f}mm")
                print(f"    Avg Wind: {df['Wind_Speed_Max_kmh'].mean():.1f}km/h")

            return df

        except Exception as e:
            print(f"  ✗ Error fetching weather: {e}")
            return None
        
    def fetch_all_zones_weather(self, start_date='2012-01-01', 
                               end_date='2017-12-31', verbose=True):
        """
        Fetch weather for all four fire zones

        Returns: Combined DataFrame with all zones
        """

        if verbose:
            print("\n" + "="*70)
            print("CARIBOO FIRE CENTRE - FETCHING ALL ZONES WEATHER")
            print("="*70)

        all_weather = []

        for zone_name in sorted(self.fire_zones.keys()):
            weather = self.fetch_weather(zone_name, start_date, end_date, verbose=verbose)
            if weather is not None:
                all_weather.append(weather)

        if not all_weather:
            print("✗ Failed to fetch any weather data")
            return None

        combined = pd.concat(all_weather, ignore_index=True)

        if verbose:
            print(f"\n{'='*70}")
            print(f"✓ Total: {len(combined)} records across {len(all_weather)} zones")
            print(f"{'='*70}\n")

        return combined


In [17]:
carib= CaribooWildfirePipeline()

In [18]:
zone= carib.classify_fire_zone(latitude=52.14, longitude=-122.14)
weather_dt= carib.fetch_all_zones_weather(start_date='2017-01-01', end_date='2017-12-31')


CARIBOO FIRE CENTRE - FETCHING ALL ZONES WEATHER

──────────────────────────────────────────────────────────────────────
Fetching Weather: 100 Mile House Fire Zone
──────────────────────────────────────────────────────────────────────
  Location: Grassland-pine mix, lower elevation
  Coordinates: (51.6403, -121.2957)
  Elevation: 960m
  Fire Season: May-September
  Date Range: 2017-01-01 to 2017-12-31
  ✗ Error fetching weather: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=51.6403&longitude=-121.2957&start_date=2017-01-01&end_date=2017-12-31&daily=temperature_2m_max&daily=temperature_2m_mean&daily=temperature_2m_min&daily=relative_humidity_2m_max&daily=relative_humidity_2m_mean&daily=relative_humidity_2m_min&daily=precipitation_sum&daily=wind_speed_10m_max&daily=pressure_msl_max&daily=dew_point_2m_mean&temperature_unit=celsius&wind_speed_unit=kmh&precipitation_unit=mm&timezone=America%2FVancouver

─────────────────────────────────

In [15]:
weather_dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                1460 non-null   datetime64[ns]
 1   Temp_Max_C          1460 non-null   float64       
 2   Temp_Mean_C         1460 non-null   float64       
 3   Temp_Min_C          1460 non-null   float64       
 4   Humidity_Max_Pct    1460 non-null   int64         
 5   Humidity_Mean_Pct   1460 non-null   int64         
 6   Humidity_Min_Pct    1460 non-null   int64         
 7   Precip_mm           1460 non-null   float64       
 8   Wind_Speed_Max_kmh  1460 non-null   float64       
 9   Pressure_Max_hPa    1460 non-null   float64       
 10  Dew_Point_Mean_C    1460 non-null   float64       
 11  Fire_Zone           1460 non-null   object        
 12  Elevation_m         1460 non-null   int64         
dtypes: datetime64[ns](1), float64(7), int64(4), obje